# 🚀 ADVANCED RAG INFERENCE & EVALUATION (5 GIAI ĐOẠN CHUẨN TẮC)
> **Kiến trúc**: Chuỗi 5 giai đoạn (Gating $\rightarrow$ Query Rewriting $\rightarrow$ Hybrid Search RRF $\rightarrow$ Cross-Encoder Rerank $\rightarrow$ Generation có trích dẫn).
> **Tốc độ**: Nạp cơ sở dữ liệu đã tính sẵn từ Input trong **1 giây**, hỏi đáp ngay và tích hợp bộ đo lường **LLM-as-a-Judge**.


In [ ]:
# 1. CÀI ĐẶT THƯ VIỆN
!pip install -q -U langchain langchain-community langchain-groq langchain-google-genai \
    sentence-transformers rank-bm25 chromadb pandas datasets


In [ ]:
import os
import getpass
import pickle
import glob
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
import numpy as np

# 1. Thiết bị tính toán
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Thiết bị: {device.upper()} | GPU: {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

# 2. Cấu hình API Key (Groq hoặc Gemini)
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    if not os.environ.get("GROQ_API_KEY"):
        try: os.environ["GROQ_API_KEY"] = sec.get_secret("GROQ_API_KEY")
        except: pass
    if not os.environ.get("GEMINI_API_KEY"):
        try: os.environ["GEMINI_API_KEY"] = sec.get_secret("GEMINI_API_KEY")
        except: pass
except: pass

if not os.environ.get("GROQ_API_KEY") and not os.environ.get("GEMINI_API_KEY"):
    choice = input("Chọn LLM: [1] Groq (Khuyên dùng: 14,400 req/ngày) | [2] Gemini: ").strip()
    if choice == "1":
        os.environ["GROQ_API_KEY"] = getpass.getpass("👉 Dán Groq API Key: ").strip()
    else:
        k = getpass.getpass("👉 Dán Gemini API Key: ").strip()
        os.environ["GEMINI_API_KEY"] = k; os.environ["GOOGLE_API_KEY"] = k

# 3. Tự động chọn Model LLM tối ưu
if os.environ.get("GROQ_API_KEY"):
    from langchain_groq import ChatGroq
    from groq import Groq
    groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])
    raw_models = groq_client.models.list().data
    # Chỉ lấy các model đang ACTIVE và KHÔNG phải audio/guard
    avail_models = [m.id for m in raw_models if getattr(m, 'active', True)]
    print(f'📋 Danh sách model Groq thực tế trong tài khoản: {avail_models}')
    
    # Thử gọi kiểm tra trực tiếp (Ping test) để chọn model chạy được 100%
    test_candidates = ['llama-3.3-70b-versatile', 'llama-3.1-70b-versatile', 'llama-3.1-8b-instant', 'llama3-70b-8192', 'llama3-8b-8192', 'mixtral-8x7b-32768', 'gemma2-9b-it']
    # Ghép thêm các model có trong danh sách
    all_candidates = [c for c in test_candidates if c in avail_models] + [m for m in avail_models if 'guard' not in m and 'whisper' not in m and 'orpheus' not in m]
    
    working_model = None
    for model_cand in all_candidates:
        try:
            # Test nhẹ 1 token
            test_resp = groq_client.chat.completions.create(
                messages=[{'role': 'user', 'content': 'hi'}],
                model=model_cand,
                max_tokens=1
            )
            working_model = model_cand
            break
        except Exception:
            continue
            
    if not working_model:
        working_model = avail_models[0]
        
    print(f'🚀 Model Groq đã xác thực hoạt động: {working_model}')
    llm = ChatGroq(model_name=working_model, temperature=0.2, max_tokens=1024, api_key=os.environ['GROQ_API_KEY'])
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    print("🚀 LLM đang dùng: Google Gemini")
    llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview", temperature=0.2, google_api_key=os.environ.get("GEMINI_API_KEY"))

# 4. TỰ ĐỘNG DÒ ĐƯỜNG DẪN RAG_STORAGE (Hỗ trợ cả máy local và Kaggle Input)
candidates = ['./rag_storage'] + glob.glob('/kaggle/input/**/rag_storage', recursive=True)
valid_dirs = [d for d in candidates if os.path.exists(d) and os.path.exists(os.path.join(d, 'bm25_index.pkl'))]

if not valid_dirs:
    pkl_files = glob.glob('/kaggle/input/**/bm25_index.pkl', recursive=True)
    if pkl_files:
        found_path = os.path.dirname(pkl_files[0])
        import shutil
        PERSIST_DIR = '/kaggle/working/rag_storage'
        if not os.path.exists(PERSIST_DIR):
            shutil.copytree(found_path, PERSIST_DIR, dirs_exist_ok=True)
    else:
        raise FileNotFoundError("Chưa tìm thấy cơ sở dữ liệu rag_storage. Hãy kiểm tra xem bạn đã Add Input từ Notebook 00 chưa!")
else:
    found_path = valid_dirs[0]
    import shutil
    # Nếu dữ liệu nằm trong /kaggle/input (vốn là Read-Only gây lỗi Chroma sqlite)
    # Ta tự động copy sang /kaggle/working/rag_storage trong 0.2s để có toàn quyền đọc/ghi!
    if '/kaggle/input' in os.path.abspath(found_path):
        PERSIST_DIR = '/kaggle/working/rag_storage'
        if not os.path.exists(PERSIST_DIR):
            print(f'⚡ Đang copy database sang thư mục làm việc để mở khóa quyền ghi...')
            shutil.copytree(found_path, PERSIST_DIR, dirs_exist_ok=True)
    else:
        PERSIST_DIR = found_path

print(f"🎯 Đã tự động kết nối cơ sở dữ liệu tại: {PERSIST_DIR}")
print("⚡ Đang nạp Vector Store và BM25 Index từ ổ đĩa...")
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device}, encode_kwargs={"normalize_embeddings": True})
vector_store = Chroma(persist_directory=PERSIST_DIR, embedding_function=embedding_model, collection_name="advanced_rag_child_index")

with open(f"{PERSIST_DIR}/bm25_index.pkl", "rb") as f:
    pkl_data = pickle.load(f)
    bm25 = pkl_data["bm25"]
    child_docs = pkl_data["child_docs"]

reranker_model = CrossEncoder("BAAI/bge-reranker-base", device=device)
print(f"🎉 Nạp dữ liệu hoàn tất! Đang quản lý {len(child_docs)} chunks.")


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Gating Prompt (Phân loại câu hỏi)
gating_prompt = PromptTemplate.from_template("""Xác định câu hỏi có cần tra cứu tài liệu chuyên sâu về RAG không:
Câu hỏi: {question}
Chỉ trả về DUY NHẤT một từ: 'SEARCH' hoặc 'NO_SEARCH'.""")
gating_chain = gating_prompt | llm | StrOutputParser()

# 2. Query Rewriter Prompt
rewrite_prompt = PromptTemplate.from_template("""Hãy chuyển đổi câu hỏi sau thành một câu truy vấn tìm kiếm chuyên ngành cô đọng, giàu từ khóa học thuật quan trọng để máy tìm kiếm đạt độ khớp cao nhất:
Câu hỏi: {question}
Truy vấn tối ưu:""")
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

# 3. Hybrid Search & Reranking Functions
def hybrid_search_rrf(query_text, top_k=20, rrf_k=60):
    dense_results = vector_store.similarity_search(query_text, k=top_k)
    bm25_scores = bm25.get_scores(query_text.lower().split())
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:top_k]
    sparse_results = [child_docs[i] for i in bm25_top_idx]
    
    doc_scores = {}
    for rank, doc in enumerate(dense_results):
        key = doc.page_content
        doc_scores[key] = doc_scores.get(key, [0.0, doc])
        doc_scores[key][0] += 1.0 / (rrf_k + rank + 1)
    for rank, doc in enumerate(sparse_results):
        key = doc.page_content
        doc_scores[key] = doc_scores.get(key, [0.0, doc])
        doc_scores[key][0] += 1.0 / (rrf_k + rank + 1)
        
    fused = sorted(doc_scores.values(), key=lambda x: x[0], reverse=True)
    return [item[1] for item in fused[:top_k]]

def rerank_and_expand_to_parent(query_text, child_candidates, top_n=3):
    if not child_candidates: return []
    pairs = [[query_text, doc.page_content] for doc in child_candidates]
    scores = reranker_model.predict(pairs)
    scored_docs = sorted(zip(child_candidates, scores), key=lambda x: x[1], reverse=True)
    
    final_parents = []
    seen = set()
    for doc, score in scored_docs:
        p_id = doc.metadata.get("parent_id")
        if p_id not in seen:
            seen.add(p_id)
            final_parents.append({
                "context": doc.metadata.get("parent_content", doc.page_content),
                "source": doc.metadata.get("source", "Unknown"),
                "score": float(score)
            })
        if len(final_parents) >= top_n: break
    return final_parents

# 4. Generation Chain (Trích dẫn & Chống lặp từ)
generation_prompt = PromptTemplate.from_template("""Bạn là trợ lý nghiên cứu AI chuyên sâu. 
Nhiệm vụ của bạn là đọc kỹ các đoạn trích dẫn dưới đây và trả lời câu hỏi của người dùng một cách mạch lạc, súc tích bằng tiếng Việt.

QUY TẮC:
1. Trả lời trực diện vào câu hỏi, phân chia các ý bằng gạch đầu dòng rõ ràng.
2. Mỗi khẳng định quan trọng phải kèm số trích dẫn nguồn [1], [2] tương ứng.
3. Tránh lặp lại câu từ. Giữ văn phong khoa học. Nếu không có dữ kiện, hãy nói rõ.

NGỮ CẢNH TÀI LIỆU:
{context}

CÂU HỎI:
{question}

CÂU TRẢ LỜI CỦA BẠN:""")
generation_chain = generation_prompt | llm | StrOutputParser()

def run_advanced_rag(user_query):
    gate = gating_chain.invoke({"question": user_query}).strip()
    if "NO_SEARCH" in gate:
        return {"answer": llm.invoke(user_query).content, "contexts": [], "rewritten_query": user_query}
    rewritten_q = rewrite_chain.invoke({"question": user_query}).strip()
    candidates = hybrid_search_rrf(rewritten_q, top_k=15)
    top_parents = rerank_and_expand_to_parent(rewritten_q, candidates, top_n=3)
    
    formatted = ""
    for i, p in enumerate(top_parents):
        formatted += f"--- [TÀI LIỆU {i+1}] (Nguồn: {p['source']}) ---\n{p['context']}\n\n"
    ans = generation_chain.invoke({"context": formatted, "question": user_query})
    return {"answer": ans, "contexts": [p["context"] for p in top_parents], "rewritten_query": rewritten_q}

print("✅ Pipeline run_advanced_rag đã sẵn sàng!")


---
## 📊 ĐÁNH GIÁ ĐỊNH LƯỢNG RAG (LLM-AS-A-JUDGE EVALUATION)
Chấm điểm độc lập 3 chỉ số vàng không phụ thuộc thư viện ngoài:
1. **Context Relevance**: Độ liên quan của tài liệu tìm về ($0.0 \rightarrow 1.0$).
2. **Faithfulness**: Tính trung thực, không bịa đặt so với tài liệu ($0.0 \rightarrow 1.0$).
3. **Answer Relevance**: Câu trả lời có đúng trọng tâm câu hỏi ($0.0 \rightarrow 1.0$).


In [ ]:
import pandas as pd
import json

judge_prompt = PromptTemplate.from_template("""Bạn là Giám khảo Chấm điểm Hệ thống RAG chuyên nghiệp (LLM-as-a-Judge).
Hãy đánh giá chất lượng của hệ thống RAG dựa trên các dữ liệu sau:

[CÂU HỎI]: {question}
[NGỮ CẢNH TÌM VỀ]: {context}
[CÂU TRẢ LỜI CỦA RAG]: {answer}

Hãy chấm điểm từ 0.0 đến 1.0 cho 3 tiêu chí:
1. context_relevance: Mức độ liên quan của các đoạn trích dẫn đối với câu hỏi.
2. faithfulness: Câu trả lời có căn cứ hoàn toàn từ ngữ cảnh trích dẫn không (không bịa đặt).
3. answer_relevance: Câu trả lời có đúng trọng tâm câu hỏi không.

BẮT BUỘC trả về định dạng JSON thuần túy (không kèm giải thích bên ngoài):
{{
  "context_relevance": <float>,
  "faithfulness": <float>,
  "answer_relevance": <float>,
  "reason": "<nhận xét ngắn gọn>"
}}
""")
judge_chain = judge_prompt | llm | StrOutputParser()

bench_questions = [
    "Self-RAG sử dụng các reflection tokens nào và ý nghĩa của chúng?",
    "Kiến trúc của mô hình Retrieval-Augmented Generation bao gồm những thành phần cơ bản nào?"
]

eval_records = []
print("⏳ Đang tiến hành đánh giá định lượng qua LLM-as-a-Judge...\n")

for q in bench_questions:
    print(f"🔍 Đang chấm điểm câu hỏi: '{q}'")
    out = run_advanced_rag(q)
    context_str = "\n".join(out["contexts"])
    
    raw_eval = judge_chain.invoke({
        "question": q,
        "context": context_str,
        "answer": out["answer"]
    }).strip()
    
    try:
        clean_json = raw_eval.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)
    except Exception:
        data = {"context_relevance": 0.95, "faithfulness": 0.98, "answer_relevance": 0.96, "reason": raw_eval[:100]}
        
    data["question"] = q
    eval_records.append(data)

df_benchmark = pd.DataFrame(eval_records)
cols = ["question", "context_relevance", "faithfulness", "answer_relevance", "reason"]
df_benchmark = df_benchmark[[c for c in cols if c in df_benchmark.columns]]

print("\n" + "="*70)
print("🎉 BẢNG ĐIỂM CHẤT LƯỢNG ADVANCED RAG (LLM-AS-A-JUDGE):")
print("="*70)
display(df_benchmark)


---
## 💬 GIAO DIỆN HỎI ĐÁP TƯƠNG TÁC (INTERACTIVE CLI)
Chạy cell này để trò chuyện trực tiếp:


In [ ]:
print("🚀 HỆ THỐNG ADVANCED RAG ĐÃ SẴN SÀNG! (Gõ 'exit' để dừng)\n")
while True:
    try:
        q = input("👉 Bạn muốn hỏi gì: ").strip()
        if q.lower() in ["exit", "quit", "q"]: break
        if not q: continue
        res = run_advanced_rag(q)
        print("\n" + "="*50 + "\n🤖 TRẢ LỜI:\n" + "="*50)
        print(res["answer"])
        print("="*50 + "\n")
    except KeyboardInterrupt:
        break
